In [28]:
import gymnasium as gym
from gymnasium.spaces import Dict, MultiDiscrete, Box
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
import numpy as np
from scipy.signal import convolve2d
from typing import Literal
from pathlib import Path
from src.config import (
    LAND_COVER_LABELS, PROTECTED_CLASSES,
    N_CLASSES, N_PIXELS_PER_CELL, SAMPLE_SIZE, GRID_KWARGS, SEED, data_dir, log_dir, model_dir
) # ECONOMIC_VALUES
from src.utils import minmax_normalize, get_logger
import wandb
from wandb.integration.sb3 import WandbCallback

logger = get_logger(__name__)

# ── Mock ET cost per land-cover class (mm / year) ──────────────────────
# Higher values → more water consumption → higher environmental cost
ET_COST = {
    1:  100,   # Water        – minimal (surface evaporation counted elsewhere)
    2:  600,   # Trees        – moderate transpiration
    4:  100,   # Flooded      – similar to open water
    5:  850,   # Crops        – irrigation-heavy
    7:  200,   # Built Area   – low (impervious surface)
    8:  50,    # Bare Ground  – very low
    9:  0,     # Snow/Ice     – negligible
    10: 0,     # Clouds       – N/A
    11: 400,   # Rangeland    – moderate
}
ECONOMIC_VALUES = {
    1:  554,    # Water        – fisheries + ecosystem services
    2:  325,    # Trees/Forest – firewood, carbon, water regulation
    4:  554,    # Flooded/Wetlands – same as water/wetlands valuation
    5:  400,    # Crops        – rainfed maize midpoint ($300-$500)
    7:  2000,   # Built Area   – urban real-estate (constraint: very high)
    8:  25,     # Bare Ground  – minimal value
    9:  0,      # Snow/Ice     – no economic value
    10: 0,      # Clouds       – no economic value (masked)
    11: 75,     # Rangeland    – low-intensity grazing
}

norm_et_cost = minmax_normalize(ET_COST)
norm_eco_values = minmax_normalize(ECONOMIC_VALUES)

# Build dense arrays for vectorised reward computation
ECO_PER_CLASS = np.zeros(N_CLASSES, dtype=np.float32)
ET_PER_CLASS  = np.zeros(N_CLASSES, dtype=np.float32)
for _cls, _val in norm_eco_values.items():
    if _cls < N_CLASSES:
        ECO_PER_CLASS[_cls] = _val
for _cls, _val in norm_et_cost.items():
    if _cls < N_CLASSES:
        ET_PER_CLASS[_cls] = _val

# Modifiable (non-protected) land-cover classes
MODIFIABLE_CLASSES = sorted(set(range(N_CLASSES)) - PROTECTED_CLASSES)
N_MOD = len(MODIFIABLE_CLASSES)

print(f"All classes       : {list(LAND_COVER_LABELS.items())}")
print(f"Protected classes : {PROTECTED_CLASSES}")
print(f"Modifiable classes: {MODIFIABLE_CLASSES}  (n={N_MOD})")
print(f"Pixels per cell   : {N_PIXELS_PER_CELL}")
print(f"Grid size per ep  : {SAMPLE_SIZE}×{SAMPLE_SIZE}")

All classes       : [(1, 'Water'), (2, 'Trees'), (4, 'Flooded'), (5, 'Crops'), (7, 'Built Area'), (8, 'Bare Ground'), (9, 'Snow/Ice'), (10, 'Clouds'), (11, 'Rangeland')]
Protected classes : frozenset({1, 3, 4, 6, 9, 10})
Modifiable classes: [0, 2, 5, 7, 8, 11]  (n=6)
Pixels per cell   : 25
Grid size per ep  : 10×10


In [25]:
ECO_PER_CLASS

array([0.    , 0.277 , 0.1625, 0.    , 0.277 , 0.2   , 0.    , 1.    ,
       0.0125, 0.    , 0.    , 0.0375], dtype=float32)

In [26]:
ET_PER_CLASS

array([0.    , 0.1176, 0.7059, 0.    , 0.1176, 1.    , 0.    , 0.2353,
       0.0588, 0.    , 0.    , 0.4706], dtype=float32)

## Env Setup Demo

In [33]:
class LandUseEnv(gym.Env):
    """
    RL Environment for land-use allocation optimization.

    State: A square grid where each cell has quantities of N_CLASSES land-cover types as features
    Action: Flattened action containing cell coordinates and delta changes
    Reward: Change in total value = Σ(proportion_k × (ECO_k - ET_k))
    """
    metadata = {"render_modes": []}

    def __init__(self, size=10, max_steps=500, et_decrease_tolerance=0.01, delta_boundary=1, 
                 split: Literal['train_indices', 'test_indices'] = 'train_indices',
                 add_spatial_reward=True, lambda_cont=0.5, lambda_buf=2.0):
        """
        Args:
            size: Grid dimension (default 10×10 cells)
            max_steps: Maximum steps per episode
            delta_boundary: Maximum absolute change per land-use class (-delta_boundary to +delta_boundary)
        """
        super().__init__()
        self.size = size
        self.max_steps = max_steps
        self.et_decrease_tolerance = et_decrease_tolerance
        self.delta_boundary = delta_boundary
        self.data = np.load(Path(data_dir, "processed", "rl_dataset.npz"))
        self.indices = self.data[split]
        self.initial_total_et  = 0.0
        self.add_spatial_reward = add_spatial_reward
        self.lambda_cont = lambda_cont
        self.lambda_buf = lambda_buf

        # observation space
        self.observation_space = Box(
            low=0, 
            high=GRID_KWARGS['grid_size']**2, 
            shape=(size, size, N_CLASSES), 
            dtype=int
        )

        # flattened action space in order to adapt the algorithm requirement, i.e. [cell_row, cell_col, delta_s_0, delta_s_1, ..., delta_s_11]
        # - cell_row, cell_col: 0 to size-1
        # - delta_s_i: map from {-delta_boundary, ..., 0, ..., +delta_boundary} to discrete indices
        delta_choices = 2 * delta_boundary + 1  # e.g., for boundary=1: {-1,0,1} = 3 choices
        self.action_space = MultiDiscrete([size, size] + [delta_choices] * N_CLASSES)
        
        # internal state
        self.state = None  # shape: (size, size, N_CLASSES)
        self.step_count = 0
        self.prev_total_value = 0.0
        
    def _decode_action(self, action):
        """
        Decode flattened action into cell coordinates and delta_s.
        
        Args:
            action: array of shape (14,) from MultiDiscrete space
        
        Returns:
            dict with 'cell_coords' and 'delta_s'
        """
        cell_coords = action[:2]  # [row, col]
        # map discrete indices back to {-delta_boundary, ..., +delta_boundary}
        delta_s = action[2:] - self.delta_boundary  # e.g., {0,1,2} -> {-1,0,1}
        return {
            'cell_coords': cell_coords,
            'delta_s': delta_s
        }
        
    def _compute_total_value(self):
        """
        Compute total value = Σ_cells Σ_classes (fraction_k × (ECO_k + ET_k))

        Args:
            state: (size, size, N_CLASSES) array of land-cover quantities
        Returns:
            float: Total economic value plus ET cost
        """
        # vectorized: (size, size, N_CLASSES) * (N_CLASSES,) → (size, size, N_CLASSES)
        net_values = self.state * (ECO_PER_CLASS + ET_PER_CLASS)
        if self.add_spatial_reward:
            spatial_reward, contiguity_bonus, buffer_penalty = self._compute_spatial_modifiers()
        else:
            spatial_reward = contiguity_bonus = buffer_penalty = 0
        total_values = float(net_values.sum()) + spatial_reward
        return total_values, spatial_reward, contiguity_bonus, buffer_penalty
        
    def _compute_total_et(self) -> float:
        return float((self.state.reshape(-1, N_CLASSES) @ ET_PER_CLASS).sum())

    def _compute_spatial_modifiers(self):
        """
        Computes spatial rewards based on contiguity and buffer zones.
        
        1. For contiguity bonus, reward the agent when cells with a high fraction 
        of a specific class (e.g. trees) are placed directly adjacent to other cells 
        with a high fraction of that same class.
        
        2. For buffer penalty, penalize the agent if it high-impact classes—such as 
        crop or built area in the cell directly adjacent to vulnerable classes, such 
        as water or flooded.
        """
        fractions = self.state / 25.0 
        
        # use simple kernel to look at immediate up/down/left/right neighbors
        kernel = np.array([[0, 1, 0],
                           [1, 0, 1],
                           [0, 1, 0]])
        
        # ---------------------------------------------------------
        # 1. contiguity bonus (e.g., for trees)
        # ---------------------------------------------------------
        trees = fractions[:, :, 2]
        
        # convolve2d sums up the tree fractions in adjacent cells
        tree_neighbors = convolve2d(trees, kernel, mode='same', boundary='fill', fillvalue=0)
        
        # multiply the cell's own tree fraction by its neighbors' tree fraction.
        # Note: this guarantees a high bonus ONLY if both the cell and its neighbors have trees.
        contiguity_bonus = np.sum(trees * tree_neighbors)
        
        # ---------------------------------------------------------
        # 2. buffer penalty (e.g. for crops(5) / built area(7) near water(1) / flooded(4))
        # ---------------------------------------------------------
        water_bodies = fractions[:, :, 1] + fractions[:, :, 4]
        high_impact = fractions[:, :, 5] + fractions[:, :, 7]

        water_neighbors = convolve2d(water_bodies, kernel, mode='same', boundary='fill', fillvalue=0)
        buffer_penalty = np.sum(high_impact * water_neighbors)
        
        # ---------------------------------------------------------
        # total spatial reward
        # ---------------------------------------------------------
        spatial_reward = (self.lambda_cont * contiguity_bonus) - (self.lambda_buf * buffer_penalty)
        return float(spatial_reward), contiguity_bonus, buffer_penalty

    def _get_obs(self, r_idx, c_idx) -> np.ndarray:
        self.state = self.data['pixel_counts'][r_idx: r_idx + self.size, c_idx: c_idx + self.size]
        return self.state

    def _apply_action(self, action):
        """
        Apply action with constraint to ensure sum of changes in the vector equal to zero
        
        Args:
            action: flattened action from MultiDiscrete space
        """
        # Decode the action
        decoded = self._decode_action(action)
        delta_s = decoded['delta_s']
        cell_coords = decoded['cell_coords']
        
        pos_mask = delta_s > 0
        neg_mask = delta_s < 0
        pos_sum = np.sum(delta_s[pos_mask])
        neg_sum = np.abs(np.sum(delta_s[neg_mask]))
        
        # if the agent only asked to increase or only asked to decrease, no valid trade can happen
        if pos_sum == 0 or neg_sum == 0:
            return self.state
        
        target_sum = min(pos_sum, neg_sum)
        final_delta_s = np.zeros_like(delta_s, dtype=float)
        
        # scale pos values down
        final_delta_s[pos_mask] = delta_s[pos_mask] * (target_sum / pos_sum)
        
        # scale neg values up
        final_delta_s[neg_mask] = delta_s[neg_mask] * (target_sum / neg_sum)
        
        # update state with action
        action_cell_features = self.state[cell_coords[0], cell_coords[1], :].astype(float)
        new_action_cell_features = action_cell_features + final_delta_s
        
        # ensure non-negative values and clip to valid range
        new_action_cell_features = np.clip(new_action_cell_features, 0, GRID_KWARGS['grid_size']**2)
        
        self.state[cell_coords[0], cell_coords[1], :] = new_action_cell_features.astype(int)
        return self.state

    def reset(self, seed=None, options=None):
        """
        Randomly select one sample from training dataset
        """
        super().reset(seed=seed)
        idx = self.np_random.integers(len(self.indices))
        r0_idx, c0_idx = self.indices[idx]
        self.state = self._get_obs(r0_idx, c0_idx).copy()  # Make a copy to avoid modifying original data
        self.step_count = 0
        self.prev_total_value, _, _, _ = self._compute_total_value()
        self.initial_total_et = self._compute_total_et()
        info = {"total_value": self.prev_total_value, "et_increase": 0, "step": self.step_count}
        return self.state, info

    def step(self, action):
        """
        Execute action at one step

        Args:
            action: Flattened action from MultiDiscrete space

        Returns:
            obs: New state
            reward: Change in total value
            terminated: Whether episode is done
            truncated: Whether max_steps reached
            info: Metadata dict
        """   
        self._apply_action(action)
        cur_total_value, spatial_reward, contiguity_bonus, buffer_penalty = self._compute_total_value()
        reward = cur_total_value - self.prev_total_value
        self.prev_total_value = cur_total_value
        self.step_count += 1

        et_increase = (self._compute_total_et() - self.initial_total_et) / (self.initial_total_et + 1e-9)
        terminated = self.step_count >= self.max_steps or et_increase < -self.et_decrease_tolerance
        truncated = False
        info = {
            "total_value": cur_total_value, 
            "et_increase": et_increase, 
            "step": self.step_count,
            "contiguity_bonus": contiguity_bonus,
            "buffer_penalty": buffer_penalty,
            "spatial_reward": spatial_reward
        }
        return self.state, reward, terminated, truncated, info


In [5]:
logger.info("Creating environment …")
env = LandUseEnv(split="train_indices")
check_env(env, warn=True)
logger.info("Environment check passed.")

model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=3e-4,
    n_steps=1000,
    batch_size=2,
    n_epochs=1,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    tensorboard_log=str(log_dir / "ppo_landuse"),
)

model.learn(total_timesteps=10000, progress_bar=False)

save_path = model_dir / "ppo_land_use"
save_path.parent.mkdir(parents=True, exist_ok=True)
model.save(str(save_path))
logger.info(f"Model saved → {save_path}")

22:59:39 | __main__ | INFO    | Creating environment …
22:59:39 | __main__ | INFO    | Environment check passed.
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to /Users/yingyao/Desktop/Code.nosync/cs8903/CS8903-odc/log/ppo_landuse/PPO_13
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1        |
|    ep_rew_mean     | 0.329    |
| time/              |          |
|    fps             | 1438     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 1000     |
---------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 1          |
|    ep_rew_mean          | 0.272      |
| time/                   |            |
|    fps                  | 864        |
|    iterations           | 2          |
|    time_elapsed         | 2          |
|    total_timesteps      | 2000      

In [18]:
model = PPO.load(str(model_dir / "ppo_land_use"))
env = LandUseEnv(split="test_indices")

for ep in range(3):
    obs, _ = env.reset()
    total_reward, done = 0.0, False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        done = terminated or truncated
    logger.info(
        f"Episode {ep + 1}: reward={total_reward:.4f}, "
        f"ET increase={info['et_increase']:.2f} %, steps={info['step']}"
    )

23:11:37 | __main__ | INFO    | Episode 1: reward=3.8118, ET increase=0.01 %, steps=1
23:11:37 | __main__ | INFO    | Episode 2: reward=3.3037, ET increase=0.00 %, steps=1
23:11:37 | __main__ | INFO    | Episode 3: reward=3.8118, ET increase=0.01 %, steps=1


## Log metrics in WandB cloud

In [34]:
from stable_baselines3.common.callbacks import BaseCallback
import numpy as np

class SpatialMetricsCallback(BaseCallback):
    """
    Custom callback for logging spatial environment metrics from the info dict.
    """
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.contiguity_buffer = []
        self.penalty_buffer = []

    def _on_step(self) -> bool:
        # Extract the info dict from the current step
        # self.locals["infos"] is a list because SB3 vectorizes environments
        for info in self.locals.get("infos", []):
            if "contiguity_bonus" in info:
                self.contiguity_buffer.append(info["contiguity_bonus"])
            if "buffer_penalty" in info:
                self.penalty_buffer.append(info["buffer_penalty"])
        return True
        
    def _on_rollout_end(self) -> None:
        # Log the mean of the metrics at the end of every rollout
        if self.contiguity_buffer:
            self.logger.record("spatial/contiguity_bonus_mean", np.mean(self.contiguity_buffer))
            self.contiguity_buffer = []
            
        if self.penalty_buffer:
            self.logger.record("spatial/buffer_penalty_mean", np.mean(self.penalty_buffer))
            self.penalty_buffer = []

In [37]:
config = {
    "policy_type": "MlpPolicy",
    "total_timesteps": 25000,
    "learning_rate": 3e-4,
    "batch_size": 2,
    "env_name": "LakeMalawiLandUse-v0",
}

# Initialize WandB
run = wandb.init(
    project="CS8903-odc",
    config=config,
    sync_tensorboard=True,  # Crucial: This intercepts the SB3 logger
    save_code=True,
)

logger.info("Creating environment …")
env = LandUseEnv(split="train_indices", max_steps=1000)

model = PPO(
    config["policy_type"],
    env,
    verbose=1,
    learning_rate=config["learning_rate"],
    batch_size=config["batch_size"],
    tensorboard_log=f"runs/{run.id}", # Keep this so WandB can read it
)

# Combine callbacks
callbacks = [
    SpatialMetricsCallback(),
    WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
]

# Train the agent
model.learn(total_timesteps=config["total_timesteps"], callback=callbacks)
run.finish()

15:36:50 | __main__ | INFO    | Creating environment …


wandb: WARNING When using several event log directories, please call `wandb.tensorboard.patch(root_logdir="...")` before `wandb.init`


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to runs/6qb6txvi/PPO_1
---------------------------------------
| rollout/                 |          |
|    ep_len_mean           | 1e+03    |
|    ep_rew_mean           | 80.6     |
| spatial/                 |          |
|    buffer_penalty_mean   | 13.8     |
|    contiguity_bonus_mean | 0.105    |
| time/                    |          |
|    fps                   | 1761     |
|    iterations            | 1        |
|    time_elapsed          | 1        |
|    total_timesteps       | 2048     |
---------------------------------------
------------------------------------------
| rollout/                 |             |
|    ep_len_mean           | 784         |
|    ep_rew_mean           | 60.5        |
| spatial/                 |             |
|    buffer_penalty_mean   | 10.1        |
|    contiguity_bonus_mean | 0.0856      |
| time/                    |             |
|    fps   

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


global_step,▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
rollout/ep_len_mean,█▂▂▃▂▁▂▂▂▂▂▂▃
rollout/ep_rew_mean,▇▁▃▃▃▃▂▄▃▅▇▇█
spatial/buffer_penalty_mean,▅▃▃▅▃▁█▆▃▄▃▃▅
spatial/contiguity_bonus_mean,▁▁▁▇▁▁▂▂▁█▁▁▁
time/fps,█▁▁▁▁▁▁▁▁▁▁▁▁
train/approx_kl,▁█▄▂▃▃▃▃▄▂▂▂
train/clip_fraction,▁▄▇▄▇▆▆▅█▄▄▅
train/clip_range,▁▁▁▁▁▁▁▁▁▁▁▁
train/entropy_loss,▁▂▅▂▄▆▄▆▇███
+5,...


In [38]:
env = LandUseEnv(split="test_indices")

for ep in range(3):
    obs, _ = env.reset()
    total_reward, done = 0.0, False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        done = terminated or truncated
    logger.info(
        f"Episode {ep + 1}: reward={total_reward:.4f}, "
        f"ET increase={info['et_increase']:.2f} %, steps={info['step']}"
    )

15:41:44 | __main__ | INFO    | Episode 1: reward=56.7275, ET increase=0.15 %, steps=500
15:41:44 | __main__ | INFO    | Episode 2: reward=56.7275, ET increase=0.16 %, steps=500
15:41:44 | __main__ | INFO    | Episode 3: reward=70.7475, ET increase=0.03 %, steps=500


## Next Steps
1. Enhance background of project and domain interpretation in web page
2. Refine training - constrain for protected class etc.
3. Understand how exactly model performs, what the evaluation metrics mean
...
5. Visualize learned policies on real geographical data

## Playaround

In [70]:
data = np.load(Path(data_dir, "processed", "rl_dataset.npz"))
indices = data['train_indices']
indices

array([[10, 30],
       [30, 10],
       [ 0,  0],
       [40, 30],
       [20, 10],
       [10, 40],
       [20, 30],
       [ 0, 10],
       [40, 20],
       [10,  0],
       [ 0, 20],
       [20, 20],
       [30,  0],
       [ 0, 30],
       [ 0, 40],
       [40,  0],
       [30, 20]])

In [71]:
data['pixel_counts'][10:20, 30:40]

array([[[ 0,  5,  0, ...,  0,  0,  1],
        [ 0,  1,  0, ...,  0,  0,  1],
        [ 0,  0,  0, ...,  0,  0,  0],
        ...,
        [ 0, 25,  0, ...,  0,  0,  0],
        [ 0, 25,  0, ...,  0,  0,  0],
        [ 0, 25,  0, ...,  0,  0,  0]],

       [[ 0,  0,  0, ...,  0,  0,  4],
        [ 0,  7,  0, ...,  0,  0,  2],
        [ 0,  0,  0, ...,  0,  0,  0],
        ...,
        [ 0, 25,  0, ...,  0,  0,  0],
        [ 0, 25,  0, ...,  0,  0,  0],
        [ 0, 25,  0, ...,  0,  0,  0]],

       [[ 0,  0,  0, ...,  0,  0,  0],
        [ 0,  1,  0, ...,  0,  0,  0],
        [ 0,  4,  0, ...,  0,  0,  0],
        ...,
        [ 0, 25,  0, ...,  0,  0,  0],
        [ 0, 25,  0, ...,  0,  0,  0],
        [ 0, 25,  0, ...,  0,  0,  0]],

       ...,

       [[ 0,  0,  0, ...,  0,  0,  0],
        [ 0,  0,  0, ...,  0,  0,  0],
        [ 0,  0,  0, ...,  0,  0,  0],
        ...,
        [ 0, 25,  0, ...,  0,  0,  0],
        [ 0, 25,  0, ...,  0,  0,  0],
        [ 0, 25,  0, ...,  0,  0

In [56]:
grid_height = 10
grid_width = 10
num_features = 12
min_value = 0.0
max_value = 25.0

# 2. Define the Box space
# We pass scalar values for low and high, and Gymnasium will automatically 
# expand them to fit the provided shape.
grid_observation_space = Box(
    low=min_value, 
    high=max_value, 
    shape=(grid_height, grid_width, num_features), 
    dtype=int # Use np.int32 if your features are strictly integers
)
state_sample = grid_observation_space.sample()
net_values = state_sample * (ECO_PER_CLASS + ET_PER_CLASS)
net_values
float(net_values.sum())

5727.554563611746

In [47]:
grid_observation_space.shape

(10, 10, 12)

In [46]:
print(state_sample[6, 4, :])
print(state_sample[6, 4, :] + action_sample['delta_s'])

[ 2 25  4  5  5 25 24 24 20 11 25 11]
[ 11  14 -19  16 -15  30  28  37  11 -14  46  -9]


In [52]:
action_space = Dict({
    # select the (i, j) cell
    "cell_coords": MultiDiscrete([size, size]), 
    # delta change for the 12 land-use types
    "delta_s": Box(low=-1, high=1, shape=(N_CLASSES,), dtype=int)
})
action_sample = action_space.sample()
action_sample

{'cell_coords': array([4, 4]),
 'delta_s': array([-1,  1,  0,  0, -1,  1,  1,  1,  1,  0,  1, -1])}

In [45]:
cell_coords = action_sample['cell_coords']
state_sample[cell_coords[0], cell_coords[1], :]

array([ 2, 25,  4,  5,  5, 25, 24, 24, 20, 11, 25, 11])

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

def visualize_state(state, title="Land-Use Distribution"):
    """
    Visualize the land-use state as a grid showing dominant class per cell.
    
    Args:
        state: (size, size, N_CLASSES) array of fractions
        title: Plot title
    """
    size = state.shape[0]
    
    # Get dominant class per cell
    dominant_class = state.argmax(axis=-1)
    dominant_fraction = state.max(axis=-1)
    
    # Create color map from config
    from src.config import LAND_COVER_COLORS, LAND_COVER_LABELS
    colors = [LAND_COVER_COLORS.get(i, '#cccccc') for i in range(N_CLASSES)]
    cmap = ListedColormap(colors)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Left: Dominant class per cell
    im1 = axes[0].imshow(dominant_class, cmap=cmap, vmin=0, vmax=N_CLASSES-1)
    axes[0].set_title(f"{title}\nDominant Land-Cover Class")
    axes[0].set_xlabel("Column")
    axes[0].set_ylabel("Row")
    
    # Add grid
    for i in range(size+1):
        axes[0].axhline(i-0.5, color='white', linewidth=0.5, alpha=0.3)
        axes[0].axvline(i-0.5, color='white', linewidth=0.5, alpha=0.3)
    
    # Right: Fraction of dominant class
    im2 = axes[1].imshow(dominant_fraction, cmap='YlGn', vmin=0, vmax=1)
    axes[1].set_title(f"{title}\nDominant Class Fraction")
    axes[1].set_xlabel("Column")
    axes[1].set_ylabel("Row")
    plt.colorbar(im2, ax=axes[1], label='Fraction')
    
    # Add grid
    for i in range(size+1):
        axes[1].axhline(i-0.5, color='white', linewidth=0.5, alpha=0.3)
        axes[1].axvline(i-0.5, color='white', linewidth=0.5, alpha=0.3)
    
    plt.tight_layout()
    return fig

# Example: Visualize initial state
env_vis = LandUseEnv(size=10, max_steps=20)
obs, info = env_vis.reset(seed=42)
state_reshaped = obs.reshape(env_vis.size, env_vis.size, N_CLASSES)

print(f"Initial total value: {info['total_value']:.2f}")
print(f"State shape: {state_reshaped.shape}")
print(f"All cells sum to 1: {np.allclose(state_reshaped.sum(axis=-1), 1.0)}\n")

# Show land-cover distribution in one cell
print("Example cell (0,0) land-cover fractions:")
for cls in range(N_CLASSES):
    if state_reshaped[0, 0, cls] > 0.01:  # Only show non-negligible fractions
        label = LAND_COVER_LABELS.get(cls, f"Class {cls}")
        print(f"  {label:15s}: {state_reshaped[0, 0, cls]:.3f}")

fig = visualize_state(state_reshaped, "Initial State (Random)")
plt.show()

In [ ]:
# ── Greedy Policy Demo ─────────────────────────────────────────────────
# A simple greedy agent that always tries to increase high-value, low-ET classes

def greedy_policy(env):
    """
    Greedy policy: Always swap from lowest-value to highest-value modifiable class.
    
    Returns:
        action: array of shape (2 + N_CLASSES,) matching MultiDiscrete([size, size] + [delta_choices]*N_CLASSES)
                format: [row, col, encoded_delta_0, ..., encoded_delta_N_CLASSES-1]
    """
    # Compute net value per class (ECO - ET)
    net_values = ECO_PER_CLASS - ET_PER_CLASS
    
    # Among modifiable classes, find best and worst
    mod_net_values = {cls: net_values[cls] for cls in MODIFIABLE_CLASSES}
    worst_class = min(mod_net_values, key=mod_net_values.get)
    best_class = max(mod_net_values, key=mod_net_values.get)
    
    # Find a cell that has some of the worst class to swap from
    worst_amounts = env.state[:, :, worst_class]
    row, col = np.unravel_index(worst_amounts.argmax(), worst_amounts.shape)
    
    # Build delta array: decrease worst_class by 1, increase best_class by 1
    delta_s = np.zeros(N_CLASSES, dtype=int)
    delta_s[worst_class] = -1
    delta_s[best_class] = +1
    
    # Encode delta into discrete indices by adding delta_boundary offset ({-1,0,1} → {0,1,2})
    encoded_delta = delta_s + env.delta_boundary
    
    return np.array([row, col] + encoded_delta.tolist())

print("Net values per modifiable class (ECO - ET):")
for cls in MODIFIABLE_CLASSES:
    label = LAND_COVER_LABELS.get(cls, f"Class {cls}")
    net_val = ECO_PER_CLASS[cls] - ET_PER_CLASS[cls]
    print(f"  {label:15s} (class {cls:2d}): {net_val:+7.4f}  "
          f"[ECO={ECO_PER_CLASS[cls]:.4f}, ET={ET_PER_CLASS[cls]:.4f}]")

print("\n" + "="*70)
print("Running greedy episode (swap from lowest-value to highest-value class)")
print("="*70 + "\n")

env_greedy = LandUseEnv(size=10, max_steps=50)
obs, info = env_greedy.reset(seed=123)

print(f"Initial total value: {info['total_value']:.4f}\n")

total_reward = 0.0
for step in range(50):
    action = greedy_policy(env_greedy)
    obs, reward, terminated, truncated, info = env_greedy.step(action)
    total_reward += reward
    
    if step < 8 or step % 10 == 0 or terminated or truncated:
        print(f"  step {info['step']:3d} | reward={reward:+7.4f} | "
              f"total_val={info['total_value']:7.4f} | spatial={info['spatial_reward']:+.4f}")
    
    if terminated or truncated:
        break

print(f"\nEpisode finished — cumulative reward: {total_reward:+.4f}")
print(f"Final total value: {info['total_value']:.4f}")

# Visualize final state
final_state = obs.reshape(env_greedy.size, env_greedy.size, N_CLASSES)
fig = visualize_state(final_state, "Final State (Greedy Policy)")
plt.show()
